<table style="width:100%; border:none; background:none;">
  <tr style="border:none;">
    <td style="border:none; vertical-align:middle; text-align:left; width: 120px;">
      <a href="https://hubs.la/Q040tvsK0"><img src="https://raw.githubusercontent.com/peremartra/Rearchitecting-LLMs/main/Images/cover.png" width="100px" style="border-radius: 4px;"></a>
    </td>
    <td style="border:none; vertical-align:middle; text-align:left;">
      <p style="margin: 0; font-size: 14px;">
        Supplementary code for the <a href="https://hubs.la/Q040tvsK0">Rearchitecting LLMs</a> book by <a href="https://github.com/peremartra">Pere Martra</a>.<br>
        <br>
        Code repository: <a href="https://github.com/peremartra/Rearchitecting-LLMs">https://github.com/peremartra/Rearchitecting-LLMs</a>
      </p>
    </td>
  </tr>
</table>

# CH13 — Putting the Model Family to Work
## Speculative decoding and cascaded routing with S and a larger sibling

[![LinkedIn](https://img.shields.io/badge/LinkedIn-0077B5?style=flat&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/pere-martra/) [![GitHub](https://img.shields.io/badge/GitHub-100000?style=flat&logo=github&logoColor=white)](https://github.com/peremartra) [![X](https://img.shields.io/badge/X-000000?style=flat&logo=x&logoColor=white)](https://x.com/PereMartra) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-blue)](https://huggingface.co/oopere)

_____
* **Authors:** Pere Martra, Ishita Verma
* **Models:** Model **S** (small) + Model **M** (default) or **T** as the larger model
* **Colab Environment:** T4 GPU
* **Keys:**
  * Speculative decoding (Hugging Face assisted generation)
  * Cascaded routing
  * Confidence gating
* **References:**
  * [Manning Book: Rearchitecting LLMs](https://livebook.manning.com/book/rearchitecting-llms)
_____

This notebook is a demonstration, not a production benchmark. The goal is to
show how two models of the family can cooperate at inference time, not to build
a faster system. With models this small and a task this narrow, neither pattern
is expected to deliver large gains, and the notebook measures that honestly.

Two patterns, both using two models of the family:

| Pattern | Small model | Larger model | What it shows |
|---|---|---|---|
| Speculative decoding | S drafts tokens | M or T verifies | Same output as the larger model, different latency |
| Cascaded routing | S answers first | M or T on escalation | When S can be trusted and when to escalate |

The larger model is chosen with `LARGE_MODEL` (`"M"` by default, `"T"` as the
alternative). Data preparation and the evaluation harness are the same ones
used in the M and S notebooks, so the results are directly comparable.

## 1. Setup

In [1]:
!pip install -q \
    transformers==5.0.0 \
    datasets==4.0.0 \
    accelerate==1.12.0 \
    scikit-learn \
    pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 28.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


In [2]:
import json
import re
import time
import random
from collections import Counter

import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.model_selection import train_test_split

from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

print(torch.cuda.get_device_name(0))

Tesla T4


In [3]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)

## 2. Configuration

`LARGE_MODEL` selects the second member of the pair. S is always the small
model: the draft in speculative decoding and the first responder in routing.

`NUM_ASSISTANT_TOKENS` is K, the number of tokens S drafts before the larger
model verifies them. It is fixed with a constant schedule so the measured
behavior can be compared with the theoretical speedup formula.

The models are loaded in float16 because the T4 has no native bfloat16
support. The M and S notebooks evaluated in bfloat16, so a few individual
outputs may differ from the figures reported there.

In [4]:
DATASET_NAME = "Salesforce/xlam-function-calling-60k"
MODEL_REPOS = {
    "T": "oopere/qwen3-0.6b-weather-geo-specialist",
    "M": "oopere/qwen3-0.6b-weather-geo-M",
    "S": "oopere/qwen3-0.6b-weather-geo-S",
}

# --- Pair selection ---
LARGE_MODEL = "M"                 # "M" (default) or "T"

# --- Data (must match the M and S notebooks) ---
TEST_SIZE = 0.30
RANDOM_STATE = 42
MIN_EXAMPLES_PER_FUNCTION = 3
ALLOWED_FUNCTIONS = {"get_ip_zipcode", "get_city_from_zipcode", "local_weather_api"}

# --- Generation ---
MAX_NEW_TOKENS = 200
DTYPE = torch.float16

# --- Speculative decoding ---
NUM_ASSISTANT_TOKENS = 5          # K: tokens drafted per verification step
WARMUP_EXAMPLES = 3

# --- Cascaded routing ---
CONF_THRESHOLD = 0.90             # minimum token probability to trust S
THRESHOLD_SWEEP = [0.0, 0.5, 0.7, 0.8, 0.9, 0.95, 0.99]

assert LARGE_MODEL in ("M", "T"), "LARGE_MODEL must be 'M' or 'T'"

## 3. Dataset

Same curation, deduplication, and stratified split as in the M and S
notebooks. Only `test_examples` and the prompts in `test_dataset` are used
here; nothing is trained.

In [5]:
def curate_domain_dataset(dataset, allowed_functions=ALLOWED_FUNCTIONS,
                          min_examples=MIN_EXAMPLES_PER_FUNCTION):
    matched = [
        ex for ex in dataset
        if len(json.loads(ex["answers"])) == 1
        and json.loads(ex["answers"])[0]["name"] in allowed_functions
    ]
    function_counts = Counter(json.loads(ex["answers"])[0]["name"] for ex in matched)
    main_functions = {name for name, count in function_counts.items() if count >= min_examples}
    return [ex for ex in matched if json.loads(ex["answers"])[0]["name"] in main_functions]

raw_dataset = load_dataset(DATASET_NAME, split="train")
domain_examples = curate_domain_dataset(raw_dataset)
print(f"Curated domain examples: {len(domain_examples)}")

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

xlam_function_calling_60k.json: reconstructing file:   0%|          |  0.00B / 96.1MB            

xlam_function_calling_60k.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Curated domain examples: 233


In [6]:
def normalize_query(text):
    return re.sub(r"[^\w\s]", "", text.lower()).strip()

def deduplicate_by_query(examples):
    seen = set()
    unique = []
    for ex in examples:
        key = normalize_query(ex["query"])
        if key not in seen:
            seen.add(key)
            unique.append(ex)
    return unique

domain_examples = deduplicate_by_query(domain_examples)
print(f"After deduplication: {len(domain_examples)}")

After deduplication: 177


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPOS["T"])
tokenizer.padding_side = "right"
print("pad_token:", repr(tokenizer.pad_token), tokenizer.pad_token_id)
print("eos_token:", repr(tokenizer.eos_token), tokenizer.eos_token_id)

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/4.17k [00:00<?, ?B/s]

pad_token: '<|endoftext|>' 151643
eos_token: '<|im_end|>' 151645


In [8]:
def build_prompt_completion(example, tokenizer):
    tools = json.loads(example["tools"])
    answers = json.loads(example["answers"])
    user_message = {"role": "user", "content": example["query"]}
    assistant_message = {
        "role": "assistant",
        "content": None,
        "tool_calls": [{
            "type": "function",
            "function": {"name": answers[0]["name"], "arguments": answers[0]["arguments"]},
        }],
    }
    full_text = tokenizer.apply_chat_template(
        [user_message, assistant_message], tools=tools, tokenize=False, enable_thinking=False,
    )
    prompt_text = tokenizer.apply_chat_template(
        [user_message], tools=tools, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,
    )
    assert full_text.startswith(prompt_text), "prompt is not a prefix of the full rendered text"
    return {"prompt": prompt_text, "completion": full_text[len(prompt_text):]}

function_labels = [json.loads(ex["answers"])[0]["name"] for ex in domain_examples]
train_examples, test_examples = train_test_split(
    domain_examples, test_size=TEST_SIZE, stratify=function_labels,
    random_state=RANDOM_STATE,
)
print(f"Train examples: {len(train_examples)}")
print(f"Test examples:  {len(test_examples)}")

test_dataset = Dataset.from_list([build_prompt_completion(ex, tokenizer) for ex in test_examples])

Train examples: 123
Test examples:  54


## 4. Loading the pair

Both models stay resident on the GPU for the whole notebook. Together they
take under 2 GB in float16, so the pair fits comfortably on a T4.

In [9]:
def load_family_member(key):
    model = AutoModelForCausalLM.from_pretrained(MODEL_REPOS[key], dtype=DTYPE).to("cuda")
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    return model

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    emb = model.get_input_embeddings().weight.numel()
    head = 0 if model.config.tie_word_embeddings else model.get_output_embeddings().weight.numel()
    return {"total": total, "embeddings": emb + head, "non_embedding": total - emb - head}

model_s = load_family_member("S")
large_model = load_family_member(LARGE_MODEL)

for label, model in [("S", model_s), (LARGE_MODEL, large_model)]:
    params = count_parameters(model)
    print(f"{label}: {model.config.num_hidden_layers} layers, "
          f"intermediate {model.model.layers[0].mlp.gate_proj.out_features}, "
          f"total {params['total'] / 1e6:.0f}M params")

config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.14GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.29GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/278 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

S: 22 layers, intermediate 1792, total 571M params
M: 25 layers, intermediate 2304, total 645M params


## 5. Evaluation harness

The harness is copied unchanged from the M and S notebooks: `generate_tool_call`
produces the completion with greedy decoding, `parse_tool_call` extracts the
JSON inside the `<tool_call>` tags, and `call_matches` compares it against the
ground truth in strict or tolerant mode.

In [10]:
def generate_tool_call(model, tokenizer, example, max_new_tokens=200):
    tools = json.loads(example["tools"])
    messages = [{"role": "user", "content": example["query"]}]
    prompt = tokenizer.apply_chat_template(
        messages, tools=tools, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None, top_p=None, top_k=None,
        )
    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def parse_tool_call(generated_text):
    match = re.search(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", generated_text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(1))
    except json.JSONDecodeError:
        return None

In [11]:
def is_substring_equivalent(a, b):
    a_clean = re.sub(r"[^\w\s]", "", str(a).lower()).strip()
    b_clean = re.sub(r"[^\w\s]", "", str(b).lower()).strip()
    if not a_clean or not b_clean:
        return False
    if not re.search(r"[a-z]", a_clean) or not re.search(r"[a-z]", b_clean):
        return False
    if len(a_clean) < 3 or len(b_clean) < 3:
        return a_clean == b_clean
    return a_clean in b_clean or b_clean in a_clean

def is_close_coordinate(key, predicted, expected, tolerance_degrees=0.05):
    if not any(k in key.lower() for k in ["lat", "lon", "lng"]):
        return False
    try:
        return abs(float(predicted) - float(expected)) <= tolerance_degrees
    except (TypeError, ValueError):
        return False

def value_matches(key, predicted, expected, tolerant):
    if tolerant:
        try:
            if float(predicted) == float(expected):
                return True
        except (TypeError, ValueError):
            pass
        if isinstance(predicted, str) and isinstance(expected, str):
            if predicted.strip().lower() == expected.strip().lower():
                return True
            if not re.search(r"[a-z]", (predicted + expected).lower()):
                if re.sub(r"\s+", "", predicted) == re.sub(r"\s+", "", expected):
                    return True
            if is_substring_equivalent(predicted, expected):
                return True
        if is_close_coordinate(key, predicted, expected):
            return True
    return predicted == expected

def call_matches(predicted, ground_truth, tolerant):
    if predicted is None or predicted.get("name") != ground_truth["name"]:
        return False
    predicted_args = predicted.get("arguments", {})
    expected_args = ground_truth["arguments"]
    if not tolerant and set(predicted_args.keys()) != set(expected_args.keys()):
        return False
    return all(
        key in predicted_args and value_matches(key, predicted_args[key], expected_value, tolerant)
        for key, expected_value in expected_args.items()
    )

In [12]:
def evaluate_model(model, tokenizer, examples, verbose=True):
    results = []
    for example in examples:
        ground_truth = json.loads(example["answers"])[0]
        generated_text = generate_tool_call(model, tokenizer, example)
        predicted = parse_tool_call(generated_text)
        results.append({
            "generated_text": generated_text,
            "predicted": predicted,
            "ground_truth": ground_truth,
            "valid_json": predicted is not None,
            "exact_match": call_matches(predicted, ground_truth, tolerant=False),
            "tolerant_match": call_matches(predicted, ground_truth, tolerant=True),
        })
    n = len(results)
    if verbose:
        print(f"  Valid JSON:      {sum(r['valid_json'] for r in results) / n:.1%}")
        print(f"  Exact match:     {sum(r['exact_match'] for r in results) / n:.1%}")
        print(f"  Tolerant match:  {sum(r['tolerant_match'] for r in results) / n:.1%}")
    return results

def breakdown_mismatches(results, verbose=True):
    no_call = wrong_function = wrong_args = 0
    for result in results:
        if result["tolerant_match"]:
            continue
        predicted = result["predicted"]
        if predicted is None:
            no_call += 1
        elif predicted.get("name") != result["ground_truth"]["name"]:
            wrong_function += 1
        else:
            wrong_args += 1
    if verbose:
        print(f"  No tool_call emitted:        {no_call}")
        print(f"  Wrong function selected:     {wrong_function}")
        print(f"  Right function, wrong args:  {wrong_args}")
    return {"no_call": no_call, "wrong_function": wrong_function, "wrong_args": wrong_args}

### Reference accuracy of each model alone

Before combining them, each model is evaluated on its own. These are the two
reference points for everything that follows: speculative decoding should
reproduce the larger model's figures, and routing should land between both.

In [13]:
print("S alone -- on test_examples:")
s_results = evaluate_model(model_s, tokenizer, test_examples)
breakdown_mismatches(s_results)

print(f"\n{LARGE_MODEL} alone -- on test_examples:")
large_results = evaluate_model(large_model, tokenizer, test_examples)
breakdown_mismatches(large_results);

S alone -- on test_examples:
  Valid JSON:      100.0%
  Exact match:     92.6%
  Tolerant match:  92.6%
  No tool_call emitted:        0
  Wrong function selected:     0
  Right function, wrong args:  4

M alone -- on test_examples:
  Valid JSON:      100.0%
  Exact match:     96.3%
  Tolerant match:  96.3%
  No tool_call emitted:        0
  Wrong function selected:     0
  Right function, wrong args:  2


## 6. Speculative decoding

S drafts `NUM_ASSISTANT_TOKENS` tokens, and the larger model checks all of them
in a single forward pass. Hugging Face implements this as *assisted
generation*: passing `assistant_model` to `generate` is enough. With greedy
decoding the verification is exact, so the output should match the larger
model's own greedy output. What can change is latency.

The draft schedule is set on S's generation config. A constant schedule keeps
K fixed, instead of the default heuristic that adapts it on the fly.

In [14]:
model_s.generation_config.num_assistant_tokens = NUM_ASSISTANT_TOKENS
model_s.generation_config.num_assistant_tokens_schedule = "constant"

def timed_generate(model, inputs, assistant_model=None):
    # Synchronize before and after so the timer measures GPU work, not the
    # asynchronous kernel launches.
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            assistant_model=assistant_model,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None, top_p=None, top_k=None,
        )
    torch.cuda.synchronize()
    return output_ids, time.perf_counter() - start

The first calls to `generate` pay for CUDA initialization and kernel
selection. Running a few examples through both paths before measuring keeps
that one-time cost out of the comparison.

In [15]:
for example in test_dataset.select(range(WARMUP_EXAMPLES)):
    warm_inputs = tokenizer(example["prompt"], return_tensors="pt").to("cuda")
    timed_generate(large_model, warm_inputs)
    timed_generate(large_model, warm_inputs, assistant_model=model_s)
print("Warmup done")

Warmup done


### One example, step by step

A single call shows the mechanics: same input, two decoding paths, and a check
that both produce the same tokens.

In [16]:
inputs = tokenizer(test_dataset[0]["prompt"], return_tensors="pt").to("cuda")
prompt_len = inputs["input_ids"].shape[-1]

baseline_ids, baseline_time = timed_generate(large_model, inputs)
spec_ids, spec_time = timed_generate(large_model, inputs, assistant_model=model_s)

gen_tokens = baseline_ids.shape[-1] - prompt_len
print(tokenizer.decode(spec_ids[0][prompt_len:], skip_special_tokens=True))
print(f"\nIdentical output:      {torch.equal(baseline_ids, spec_ids)}")
print(f"Tokens generated:      {gen_tokens}")
print(f"{LARGE_MODEL} alone:              {baseline_time * 1000:6.1f} ms")
print(f"Speculative (S -> {LARGE_MODEL}):  {spec_time * 1000:6.1f} ms")
print(f"Speedup:               {baseline_time / spec_time:.2f}x")

<tool_call>
{"name": "local_weather_api", "arguments": {"q": "London, UK", "alerts": "yes", "aqi": "yes"}}
</tool_call>

Identical output:      True
Tokens generated:      36
M alone:              1905.8 ms
Speculative (S -> M):  1949.8 ms
Speedup:               0.98x


### Measuring the acceptance rate

`generate` does not report how many draft tokens were accepted, but under
greedy decoding it can be measured directly. A draft token is accepted when it
equals the larger model's token at that position. Running S once over the
prompt plus the larger model's completion gives S's prediction at every
position, and the fraction that match is the per-token acceptance rate α.

In [17]:
def draft_acceptance_rate(draft_model, target_ids, prompt_len):
    # Teacher-forced pass: S sees the larger model's own tokens as prefix, which
    # is exactly the prefix it sees during verification under greedy decoding.
    with torch.no_grad():
        logits = draft_model(target_ids).logits
    draft_preds = logits[0, prompt_len - 1:-1].argmax(dim=-1)
    target_tokens = target_ids[0, prompt_len:]
    return (draft_preds == target_tokens).float().mean().item()

alpha = draft_acceptance_rate(model_s, baseline_ids, prompt_len)
print(f"Acceptance rate on this example: {alpha:.2f}")

Acceptance rate on this example: 1.00


### Across the whole test set

The same comparison over the 54 test examples. For each one we record whether
the outputs match, the acceptance rate, and both latencies.

In [18]:
spec_rows = []
for example in test_dataset:
    inputs = tokenizer(example["prompt"], return_tensors="pt").to("cuda")
    prompt_len = inputs["input_ids"].shape[-1]

    baseline_ids, baseline_time = timed_generate(large_model, inputs)
    spec_ids, spec_time = timed_generate(large_model, inputs, assistant_model=model_s)

    spec_rows.append({
        "identical": torch.equal(baseline_ids, spec_ids),
        "alpha": draft_acceptance_rate(model_s, baseline_ids, prompt_len),
        "tokens": baseline_ids.shape[-1] - prompt_len,
        "large_ms": baseline_time * 1000,
        "spec_ms": spec_time * 1000,
    })

spec_df = pd.DataFrame(spec_rows)
spec_df["speedup"] = spec_df["large_ms"] / spec_df["spec_ms"]

n_diff = (~spec_df["identical"]).sum()
print(f"Outputs differing from {LARGE_MODEL}'s greedy output: {n_diff} of {len(spec_df)}")
print(f"Mean acceptance rate (alpha):  {spec_df['alpha'].mean():.2f}")
print(f"Median latency {LARGE_MODEL} alone:     {spec_df['large_ms'].median():.1f} ms")
print(f"Median latency speculative:   {spec_df['spec_ms'].median():.1f} ms")
print(f"Median speedup:               {spec_df['speedup'].median():.2f}x")

Outputs differing from M's greedy output: 0 of 54
Mean acceptance rate (alpha):  1.00
Median latency M alone:     1487.6 ms
Median latency speculative:   1441.5 ms
Median speedup:               1.04x


### Why the speedup is small (or negative)

With K draft tokens, acceptance rate α, and a draft step that costs a fraction
c of a target step, the expected speedup is

$$\text{speedup} = \frac{1 - \alpha^{K+1}}{(1 - \alpha)(cK + 1)}$$

At this scale decoding is memory-bound, so the cost of a step is roughly
proportional to the weights it reads. Pruning removed Transformer blocks and
MLP width, but S keeps the full hidden size and the 151,936-token vocabulary,
so the shared embedding table weighs heavily in both models. The cell below
estimates c from parameter counts and also from the measured per-token
latency of each model alone.

In [19]:
def expected_speedup(alpha, k, c):
    return (1 - alpha ** (k + 1)) / ((1 - alpha) * (c * k + 1))

c_params = count_parameters(model_s)["total"] / count_parameters(large_model)["total"]

# Per-token latency of S alone, measured on the same prompts as the larger model.
s_ms = []
for example in test_dataset:
    inputs = tokenizer(example["prompt"], return_tensors="pt").to("cuda")
    s_ids, s_time = timed_generate(model_s, inputs)
    s_ms.append(s_time * 1000 / (s_ids.shape[-1] - inputs["input_ids"].shape[-1]))
large_ms_per_token = (spec_df["large_ms"] / spec_df["tokens"]).median()
c_measured = np.median(s_ms) / large_ms_per_token

alpha_mean = spec_df["alpha"].mean()
print(f"c from parameter counts: {c_params:.2f}")
print(f"c from measured latency: {c_measured:.2f}")
print(f"Expected speedup (alpha={alpha_mean:.2f}, K={NUM_ASSISTANT_TOKENS}):")
print(f"  with c from parameters: {expected_speedup(alpha_mean, NUM_ASSISTANT_TOKENS, c_params):.2f}x")
print(f"  with c from latency:    {expected_speedup(alpha_mean, NUM_ASSISTANT_TOKENS, c_measured):.2f}x")

c from parameter counts: 0.88
c from measured latency: 0.88
Expected speedup (alpha=1.00, K=5):
  with c from parameters: 1.10x
  with c from latency:    1.10x


The mechanism works: S drafts, the larger model verifies, and the output is
the larger model's. Whether it pays off depends on c. A useful draft needs to
be much cheaper than the target, typically a model several times smaller. Two
siblings obtained by pruning blocks and MLP width are too close in cost for
that.

## 7. Cascaded routing

Routing sends each request to S first and escalates to the larger model only
when S's answer does not pass a check. Because the output is a structured tool
call, the check can be deterministic:

1. **Syntactic integrity.** The completion must contain a parseable
   `<tool_call>` with a function name.
2. **Minimum token probability.** The least confident generated token must
   clear `CONF_THRESHOLD`. The minimum is used instead of the mean because most
   of the ~30 tokens are formulaic and sit near 1.0; a mean would hide a single
   uncertain argument value.

`generate_with_confidence` is `generate_tool_call` from the harness, extended
to return the probability of each generated token.

In [20]:
def generate_with_confidence(model, tokenizer, prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None, top_p=None, top_k=None,
            return_dict_in_generate=True,
            output_logits=True,  # raw logits, before any generation processor
        )
    gen_ids = outputs.sequences[0, inputs["input_ids"].shape[-1]:]
    probs = torch.stack([
        torch.softmax(step_logits[0].float(), dim=-1)[token_id]
        for step_logits, token_id in zip(outputs.logits, gen_ids)
    ])
    weakest = probs.argmin().item()
    return {
        "text": tokenizer.decode(gen_ids, skip_special_tokens=True),
        "min_prob": probs[weakest].item(),
        "weakest_token": tokenizer.decode(gen_ids[weakest]),
    }

In [21]:
def route_request(prompt, small_model, large_model, threshold=CONF_THRESHOLD):
    small_out = generate_with_confidence(small_model, tokenizer, prompt)
    parsed = parse_tool_call(small_out["text"])

    # Fast path: S produced a well-formed call and was confident everywhere.
    if parsed is not None and parsed.get("name") and small_out["min_prob"] >= threshold:
        return {"call": parsed, "served_by": "S", "s_min_prob": small_out["min_prob"],
                "s_weakest_token": small_out["weakest_token"]}

    large_out = generate_with_confidence(large_model, tokenizer, prompt)
    return {"call": parse_tool_call(large_out["text"]), "served_by": LARGE_MODEL,
            "s_min_prob": small_out["min_prob"],
            "s_weakest_token": small_out["weakest_token"]}

### Three requests through the router

The first is an ordinary example. The other two are the cases identified in
the S notebook where S gets the arguments wrong and M gets them right. The
question is whether S's confidence drops on those two, so the router can catch
them.

In [22]:
DEMO_QUERIES = [
    "Fetch the weather forecast for Los Angeles, CA for the next 5 days",
    "Could you provide the current weather conditions and a 5-day forecast for New York City",
]
demo_idx = [0] + [
    next(i for i, ex in enumerate(test_examples) if ex["query"].startswith(q))
    for q in DEMO_QUERIES
]

for i in demo_idx:
    routed = route_request(test_dataset[i]["prompt"], model_s, large_model)
    ground_truth = json.loads(test_examples[i]["answers"])[0]
    print("-" * 70)
    print("QUERY:        ", test_examples[i]["query"])
    print(f"S min prob:    {routed['s_min_prob']:.3f}  (token {routed['s_weakest_token']!r})")
    print("SERVED BY:    ", routed["served_by"])
    print("CALL:         ", routed["call"])
    print("GROUND TRUTH: ", ground_truth)
    print("EXACT MATCH:  ", call_matches(routed["call"], ground_truth, tolerant=False))

----------------------------------------------------------------------
QUERY:         Fetch the local weather data for London, UK, including weather alerts and air quality.
S min prob:    0.745  (token '"}}\n')
SERVED BY:     M
CALL:          {'name': 'local_weather_api', 'arguments': {'q': 'London, UK', 'alerts': 'yes', 'aqi': 'yes'}}
GROUND TRUTH:  {'name': 'local_weather_api', 'arguments': {'q': 'London, UK', 'aqi': 'yes', 'alerts': 'yes'}}
EXACT MATCH:   True
----------------------------------------------------------------------
QUERY:         Fetch the weather forecast for Los Angeles, CA for the next 5 days including air quality data in English.
S min prob:    0.447  (token 'tp')
SERVED BY:     M
CALL:          {'name': 'local_weather_api', 'arguments': {'q': 'Los Angeles, CA', 'aqi': 'yes', 'lang': 'en', 'num_of_days': 5}}
GROUND TRUTH:  {'name': 'local_weather_api', 'arguments': {'q': 'Los Angeles, CA', 'num_of_days': 5, 'aqi': 'yes', 'lang': 'en'}}
EXACT MATCH:   True
--------

### The operating curve

To see the effect of the threshold, S and the larger model are run once on
every test example, recording S's minimum probability, both answers, and both
latencies. Any threshold can then be applied without generating again.

In [23]:
route_rows = []
for example, raw in zip(test_dataset, test_examples):
    ground_truth = json.loads(raw["answers"])[0]

    torch.cuda.synchronize(); start = time.perf_counter()
    small_out = generate_with_confidence(model_s, tokenizer, example["prompt"])
    torch.cuda.synchronize(); s_time = time.perf_counter() - start

    torch.cuda.synchronize(); start = time.perf_counter()
    large_out = generate_with_confidence(large_model, tokenizer, example["prompt"])
    torch.cuda.synchronize(); large_time = time.perf_counter() - start

    s_call = parse_tool_call(small_out["text"])
    route_rows.append({
        "s_valid": s_call is not None and bool(s_call.get("name")),
        "s_min_prob": small_out["min_prob"],
        "s_exact": call_matches(s_call, ground_truth, tolerant=False),
        "large_exact": call_matches(parse_tool_call(large_out["text"]), ground_truth,
                                    tolerant=False),
        "s_ms": s_time * 1000,
        "large_ms": large_time * 1000,
    })

route_df = pd.DataFrame(route_rows)

In [24]:
def routing_curve(df, thresholds):
    rows = []
    for threshold in thresholds:
        served_by_s = df["s_valid"] & (df["s_min_prob"] >= threshold)
        exact = np.where(served_by_s, df["s_exact"], df["large_exact"]).mean()
        # S always runs; the larger model only runs on escalated requests.
        latency = (df["s_ms"] + (~served_by_s) * df["large_ms"]).mean()
        rows.append({
            "threshold": threshold,
            "served_by_S": f"{served_by_s.mean():.0%}",
            "exact_match": f"{exact:.1%}",
            "mean_latency_ms": round(latency, 1),
            f"latency_vs_{LARGE_MODEL}": f"{latency / df['large_ms'].mean():.2f}",
        })
    return pd.DataFrame(rows)

print(f"S alone:  exact match {route_df['s_exact'].mean():.1%}, "
      f"mean latency {route_df['s_ms'].mean():.1f} ms")
print(f"{LARGE_MODEL} alone:  exact match {route_df['large_exact'].mean():.1%}, "
      f"mean latency {route_df['large_ms'].mean():.1f} ms\n")
routing_curve(route_df, THRESHOLD_SWEEP)

S alone:  exact match 92.6%, mean latency 1583.7 ms
M alone:  exact match 96.3%, mean latency 1723.4 ms



,threshold,served_by_S,exact_match,mean_latency_ms,latency_vs_M
0,0.00,100%,92.6%,1583.7,0.92
1,0.50,94%,94.4%,1695.6,0.98
2,0.70,89%,96.3%,1812.8,1.05
3,0.80,87%,96.3%,1843.8,1.07
4,0.90,87%,96.3%,1843.8,1.07
5,0.95,87%,96.3%,1843.8,1.07
6,0.99,83%,96.3%,1964.1,1.14


This curve describes the mechanism; it is not a tuned setting. With 54 test
examples, picking the best threshold here would overfit to the test set, and S
was distilled on the training prompts, so its confidence may be optimistic. In
a real deployment the threshold is chosen on a held-out validation split.

The latency column also shows the limit of this family as a cascade. If S
costs a fraction c of the larger model and a share p of requests escalate, the
mean latency is roughly (c + p) times the larger model's. With siblings this
close in size, c is high, and every escalated request pays for both models.